# Upgrades all outdated python packages with pip
- Update all python packages
- Install trough proxy

**TODO:**
- convert verbose argument to bool
- add option for proxy
```
proxy = f'http://{user}:{pwd}@{PROXYSERVER}'
os.putenv('HTTP_PROXY', proxy)
os.putenv('HTTPS_PROXY', proxy)
```
- add tests

In [ ]:
from __future__ import print_function

from multiprocessing import Pool, cpu_count
from subprocess import PIPE, Popen

import json
import argparse
import functools

## Run a command

In [ ]:
def run_command(command: str) -> tuple[str, str]:
    """ Executes a command.

    Parameters
    ----------
    command: string
    """
    stdout, stderror = Popen(command,
                             stdout=PIPE,
                             stderr=PIPE,
                             shell=True).communicate()
    return stdout, stderror

## Collect outdated packages

In [ ]:
def collect_packages(pip_cmd: str="pip", verbose: bool=True) -> list[str]:
    """ Collect outdated packages

    Returns
    -------
    packages: list of strings
    """

    outdated_command = " ".join((pip_cmd,"list --outdated --format json"))
    stdout, stderr = run_command(outdated_command)
    
    if verbose and stdout or stderr:
        print(stdout, stderr)

    pkgs = json.loads(stdout)
    return [ p['name'] for p in pkgs ]

## Upgrade a package

In [ ]:
def upgrade_package(
    package: str,
    pip_cmd: str="pip",
    dry_run: bool=False,
    verbose: int=0
) -> None:
    """ Upgrade a package.

    Parameters
    ----------
    package: string
    """
    upgrade_command = f"{pip_cmd} install --upgrade {package}"

    if verbose:
        print(upgrade_command)

    if not dry_run:
        stdout, stderr = run_command(upgrade_command)
        if verbose > 1: 
            print(package, stdout, stderr)
        else:
            print(package, stdout)

## Main Run

In [ ]:
descr = 'upgrade outdated python packages with pip.'

parser = argparse.ArgumentParser(description=descr)
group=parser.add_mutually_exclusive_group()
group.add_argument("-3", dest="pip_cmd", action="store_const", const="pip3", help="use pip3")
group.add_argument("--pip_cmd", action="store", default="pip", help="use PIP_CMD")
parser.add_argument("--verbose", "-v", action="count", default=0)
parser.add_argument("--dry_run", "-n", action="store_true", help="get list, but don't upgrade")
parser.add_argument("--serial", "-s", action="store_true", help="upgrade in serial rather than parallel")

args = parser.parse_args()

pip_cmd = args.pip_cmd if args.pip_cmd else "pip"
    
if args.verbose>1:
    print(args)
    print("pip_cmd=%s" % pip_cmd)

packages = collect_packages(pip_cmd=pip_cmd, verbose=args.verbose)
if args.verbose:
    print("Collected: ", packages)
if not args.serial:
    if args.verbose: 
        print("parallel")
    pool = Pool(cpu_count())
    pool.map(
        functools.partial(
            upgrade_package,
            pip_cmd=pip_cmd,
            dry_run=args.dry_run,
            verbose=args.verbose
        ),
        packages
    )
    pool.close()
    pool.join()
else:
    if args.verbose>1: 
        print("serial")

    all_packages = " ".join(packages)
    upgrade_package(
        all_packages,
        pip_cmd=pip_cmd,
        dry_run=args.dry_run,
        verbose=args.verbose
    )